In [1]:
"""Feature tables for the outage models, in one file for SageMaker JupyterLab.
 
How to use (SageMaker terminal):   pip install s3fs
Then either paste this whole file into one notebook cell, or upload it and run:  %run build_features.py
Reads the cleaned tables from S3 into memory. Nothing is written anywhere.
Same code as data.py + features.py in the repo.
"""
from functools import cache
 
import pandas as pd
 
BUCKET = "s3://sagemaker-aep-outage-indiansassemble/processed"
BATCH = {
    "premise": 1, "chain": 1, "transformer": 1, "device": 1, "zone": 1,
    "damage": 2, "outage_labels": 2, "zone_damage_counts": 2, "access_labels": 2,
    "pole": 3, "line": 3,
}
 
 
@cache
def load(table):
    return pd.read_parquet(f"{BUCKET}/batch{BATCH[table]}/{table}.parquet")
 
 
EVENTS = ["feb2025", "sep2024", "other"]
DAMAGE_TYPES = ["pole", "primary_wire", "transformer", "device", "secondary"]
# History uses only the "other" events, so neither Feb 2025 nor Sep 2024 labels leak into features.
HISTORY_EVENT = "other"
 
 
def home_links():
    """premise_id -> link (its transformer, its zone, and every protective zone on its chain)."""
    p, c = load("premise"), load("chain")
    zones = set(load("zone").zone_id)
    on_chain = c.loc[c.device_loc.isin(zones), ["premise_id", "device_loc"]]
    return (pd.concat([
        on_chain.rename(columns={"device_loc": "link"}),
        p[["premise_id", "zone_id"]].rename(columns={"zone_id": "link"}),
        p[["premise_id", "transformer_loc"]].rename(columns={"transformer_loc": "link"}),
    ]).dropna().drop_duplicates())
 
 
def _hits(event):
    """Damage reports per outage link (zone or transformer) in one event."""
    d = load("damage")
    return d.loc[d.event == event].groupby("outage_loc").size()
 
 
def zone_table():
    z = load("zone").set_index("zone_id")
    f = pd.DataFrame(index=z.index)
    f["circuit_id"] = z.circuit_id
    f["customers"] = z.customers
    f["is_recloser"] = z.device_type.str.startswith("Recloser").fillna(False).astype(int)
    f["breaker_only"] = z.device_type.isna().astype(int)
 
    # line km by level x mounting
    ln = load("line")
    kind = ln.level + "_" + ln.mounting.str.lower().fillna("unknown")
    km = (ln.length_m / 1000).groupby([ln.zone_id, kind]).sum().unstack(fill_value=0)
    for col in ["primary_overhead", "primary_underground", "secondary_overhead", "secondary_underground"]:
        f[f"{col}_km"] = km.get(col, 0).reindex(f.index).fillna(0)
    total_km = f.filter(like="_km").sum(axis=1)
    f["customers_per_km"] = f.customers / (total_km + 0.1)  # town vs rural
 
    # pole condition (install year known for only ~50%, so the share known is a feature too)
    pl = load("pole").dropna(subset=["zone_id"])
    g = pl.groupby("zone_id")
    f["poles"] = g.size().reindex(f.index).fillna(0)
    f["poles_per_oh_km"] = f.poles / (f.primary_overhead_km + f.secondary_overhead_km + 0.1)
    f["pole_age_known"] = g.age_years.apply(lambda s: s.notna().mean()).reindex(f.index)
    f["pole_age_mean"] = g.age_years.mean().reindex(f.index)
    f["pole_old_share"] = g.age_years.apply(lambda s: (s.dropna() > 50).mean()).reindex(f.index)
    f["pole_wood_share"] = g.material_group.apply(lambda s: (s == "wood").mean()).reindex(f.index)
    f["pole_height_mean"] = g.height_ft.mean().astype(float).reindex(f.index)
 
    # transformers
    x = load("transformer")
    gx = x.groupby("zone_id")
    f["transformers"] = gx.size().reindex(f.index).fillna(0)
    f["oh_transformer_share"] = gx.mounting.apply(lambda s: (s == "Overhead").mean()).reindex(f.index)
 
    # protective zones between this zone and the breaker
    hl = home_links()
    zl = hl[hl.link.isin(f.index)]
    depth = zl.groupby("premise_id").size() - 1
    f["depth"] = depth.groupby(load("premise").set_index("premise_id").zone_id).max().reindex(f.index).fillna(0)
 
    f["history_damage"] = _hits(HISTORY_EVENT).reindex(f.index).fillna(0)
 
    # zones with no poles / transformers: fill with typical values (the *_known / counts carry the signal)
    for col in ["pole_age_known", "pole_old_share", "pole_wood_share", "oh_transformer_share"]:
        f[col] = f[col].fillna(0)
    for col in ["pole_age_mean", "pole_height_mean"]:
        f[col] = f[col].fillna(f[col].median())
 
    # labels
    zc = load("zone_damage_counts")
    for ev in EVENTS:
        f[f"damaged_{ev}"] = (_hits(ev).reindex(f.index).fillna(0) > 0).astype(int)
        counts = zc[zc.event == ev].set_index("zone_id")[DAMAGE_TYPES]
        for t in DAMAGE_TYPES:
            f[f"n_{t}_{ev}"] = counts[t].reindex(f.index).fillna(0)
    return f
 
 
def xf_table(zones=None):
    zones = zone_table() if zones is None else zones
    x = load("transformer").dropna(subset=["zone_id"]).set_index("transformer_loc")
    x = x[x.zone_id.isin(zones.index)]
    f = pd.DataFrame(index=x.index)
    f["circuit_id"] = x.circuit_id
    f["xf_customers"] = x.customers
    f["xf_overhead"] = (x.mounting == "Overhead").astype(int)
    f["xf_kva"] = x.kva.astype(float).fillna(x.kva.median())
    f["xf_single_phase"] = x.phase.eq("Single Phase").fillna(False).astype(int)
    # the zone's secondary wire shared out per transformer
    zf = zones.loc[x.zone_id]
    f["secondary_oh_km_per_xf"] = (zf.secondary_overhead_km / zf.transformers.clip(lower=1)).values
    for col in ["customers_per_km", "pole_age_mean", "pole_old_share", "pole_wood_share", "pole_age_known"]:
        f[col] = zf[col].values
    f["history_damage"] = _hits(HISTORY_EVENT).reindex(f.index).fillna(0)
    for ev in EVENTS:
        f[f"damaged_{ev}"] = (_hits(ev).reindex(f.index).fillna(0) > 0).astype(int)
    return f
 
 
def access_table(zones=None):
    zones = zone_table() if zones is None else zones
    d = load("damage").dropna(subset=["access_issue", "damage_cat"])
    d = d[d.xf_zone.isin(zones.index)]
    f = zones.loc[d.xf_zone, [
        "circuit_id", "customers_per_km", "primary_overhead_km", "primary_underground_km",
        "secondary_overhead_km", "poles_per_oh_km", "pole_age_mean", "pole_wood_share", "depth",
    ]].set_index(d.index)
    for t in DAMAGE_TYPES:
        f[f"is_{t}"] = (d.damage_cat == t).astype(int)
    f["event"] = d.event
    f["access_issue"] = d.access_issue.astype(int)
    return f
 
 
if __name__ == "__main__":
    zones = zone_table()
    xfs = xf_table(zones)
    acc = access_table(zones)
    print("zones", zones.shape, "| transformers", xfs.shape, "| access reports", acc.shape)
    print(zones.describe().T.round(2))
    # Optional, only if the team agrees to write to S3:
    zones.to_parquet("s3://sagemaker-aep-outage-indiansassemble/features/zone_features.parquet")
    F = "s3://sagemaker-aep-outage-indiansassemble/features/"
    xfs.to_parquet(F + "transformer_features.parquet")
    acc.to_parquet(F + "access_features.parquet")
    home_links().to_parquet(F + "home_links.parquet")
 

zones (6664, 38) | transformers (25303, 15) | access reports (1527, 16)
                           count   mean    std   min    25%    50%    75%  \
customers                 6664.0  17.65  30.85   1.0   2.00   6.00  20.00   
is_recloser               6664.0   0.04   0.21   0.0   0.00   0.00   0.00   
breaker_only              6664.0   0.02   0.14   0.0   0.00   0.00   0.00   
primary_overhead_km       6664.0   0.46   0.74   0.0   0.07   0.21   0.53   
primary_underground_km    6664.0   0.15   0.48   0.0   0.00   0.00   0.12   
secondary_overhead_km     6664.0   0.34   0.62   0.0   0.00   0.11   0.39   
secondary_underground_km  6664.0   0.20   0.58   0.0   0.02   0.04   0.13   
customers_per_km          6664.0  13.11  19.23   0.3   4.17   7.55  15.51   
poles                     6664.0  12.45  19.01   0.0   3.00   6.00  14.00   
poles_per_oh_km           6664.0  14.94  27.26   0.0   8.96  11.68  14.90   
pole_age_known            6664.0   0.43   0.38   0.0   0.00   0.33   0.83   
pole